# 🎬 ShunopsAI Autonomous Video Renderer (Google Colab T4 GPU)

**Job ID:** `mw86tahh` — *IND vs PAK U19 Future Stars Ki Jung*

This notebook renders the full multimodal video on **Google Colab Free T4 GPU** using pre-generated audio, audited keyframes, dynamic BGM mixing, and automated YouTube thumbnail generation.

> 💡 **GPU Setting**: Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** -> `Save` before running.

In [ ]:
# @title 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# @title 2. Setup Environment & Clone ShunopsAI Repository
import os, shutil

if os.path.exists('/content/ShunopsAI'):
    shutil.rmtree('/content/ShunopsAI')

!git clone https://github.com/ShunyaPulse/ShunopsAI.git /content/ShunopsAI
%cd /content/ShunopsAI

# Create Kaggle directory structure required by render.py
os.makedirs('/kaggle/input/dataset', exist_ok=True)
os.makedirs('/kaggle/working/tmp', exist_ok=True)

# Install core rendering & ML dependencies
!pip install -q -U diffusers transformers accelerate safetensors huggingface_hub ftfy opencv-python Pillow edge-tts

In [ ]:
# @title 3. Download & Mount Job mw86tahh Verified Assets (Instant ~2s)
import zipfile, urllib.request

asset_url = 'https://github.com/ShunyaPulse/ShunopsAI/releases/download/v3.0.0-mw86tahh/mw86tahh_dataset.zip'
zip_dest = '/content/mw86tahh_dataset.zip'

print('⬇️ Downloading pre-generated assets (37 MB)...')
urllib.request.urlretrieve(asset_url, zip_dest)

print('📦 Extracting assets into /kaggle/input/dataset...')
with zipfile.ZipFile(zip_dest, 'r') as z:
    z.extractall('/kaggle/input/dataset')

print('✅ Dataset successfully mounted at /kaggle/input/dataset!')
!ls -la /kaggle/input/dataset

In [ ]:
# @title 4. Run Rendering Engine
# Executes video/kernel/render.py with float32 VAE, dual-worker dispatch, and clean linear frame blending
!python video/kernel/render.py

In [ ]:
# @title 5. Preview & Download Rendered Video & Thumbnail
import os
from google.colab import files
from IPython.display import Image, display

# 1. Display generated thumbnail
thumb_path = '/kaggle/working/thumbnail_high_ctr.png'
if os.path.exists(thumb_path):
    print('🖼️ High-CTR YouTube Thumbnail:')
    display(Image(thumb_path, width=640))

# 2. Check and trigger download
video_path = '/kaggle/working/video.mp4'
if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    print(f'🎉 Video Render Succeeded! Size: {size_mb:.2f} MB')
    print('⬇️ Downloading video.mp4...')
    files.download(video_path)
    if os.path.exists(thumb_path):
        files.download(thumb_path)
else:
    print('❌ video.mp4 not found in /kaggle/working. Check execution log above.')